In [ ]:
# ── Dépendances ──────────────────────────────────────────────────────────────
# pip install sentence-transformers datasets pillow numpy scikit-learn tqdm matplotlib seaborn

import warnings
warnings.filterwarnings('ignore')

import os, json, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from collections import defaultdict
from tqdm.notebook import tqdm
from PIL import Image

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# ── Modèle local CLIP (sentence-transformers) ─────────────────────────────────
# clip-ViT-B-32 : modèle multimodal texte+image, zéro API, tourne en CPU
# Principe identique à Jina CLIP v2 — même espace vectoriel texte/image
from sentence_transformers import SentenceTransformer

CLIP_MODEL_NAME = 'clip-ViT-B-32'
EMBED_DIM       = 512

print(f'Chargement du modèle local : {CLIP_MODEL_NAME}')
clip_model = SentenceTransformer(CLIP_MODEL_NAME)
print(f'Modèle chargé | Dimension : {EMBED_DIM}')
print('Aucune API externe — compatible on-premise / souveraineté GxP NOVAGEN')


In [ ]:
from datasets import load_dataset

print('Chargement du corpus ViDoRe V3 Pharmaceuticals...')
# cast_columns=False + keep_in_memory=False : les images ne sont PAS décodées en RAM
# Elles restent en bytes bruts et sont converties à la volée lors de l'encodage
corpus_ds  = load_dataset('vidore/vidore_v3_pharmaceuticals', 'corpus',  split='test')
queries_ds = load_dataset('vidore/vidore_v3_pharmaceuticals', 'queries', split='test')
qrels_ds   = load_dataset('vidore/vidore_v3_pharmaceuticals', 'qrels',   split='test')

df_queries = queries_ds.to_pandas()
df_qrels   = qrels_ds.to_pandas()

# corpus : on garde UNIQUEMENT les colonnes légères (pas la colonne image)
# Les images seront accédées par index depuis corpus_ds directement (lazy)
df_corpus = corpus_ds.to_pandas().drop(columns=['image'], errors='ignore')
df_corpus['corpus_idx'] = range(len(df_corpus))  # index pour accès lazy

print(f'Corpus  : {len(df_corpus):,} pages (images NON chargées en RAM)')
print(f'Queries : {len(df_queries):,} requêtes')
print(f'Qrels   : {len(df_qrels):,} paires de pertinence')
print(f'Colonnes corpus : {df_corpus.columns.tolist()}')


In [ ]:
def to_pil(img_field) -> Image.Image:
    """Convertit un champ image HuggingFace (dict, bytes, PIL) en PIL.Image RGB."""
    if img_field is None:
        return Image.new('RGB', (400, 300), color='white')
    if isinstance(img_field, Image.Image):
        return img_field.convert('RGB')
    if isinstance(img_field, dict):
        raw = img_field.get('bytes') or img_field.get('data')
        if raw:
            return Image.open(io.BytesIO(raw)).convert('RGB')
        path = img_field.get('path')
        if path:
            return Image.open(path).convert('RGB')
    if isinstance(img_field, bytes):
        return Image.open(io.BytesIO(img_field)).convert('RGB')
    try:
        return Image.fromarray(np.uint8(img_field)).convert('RGB')
    except Exception:
        return Image.new('RGB', (400, 300), color='white')


def embed_images_clip(images: list, model, batch_size: int = 32) -> np.ndarray:
    """
    Encode une liste de PIL Images en embeddings visuels via CLIP local.
    Retourne un tableau numpy (n_images, EMBED_DIM), normalisé L2.
    """
    all_embeddings = []
    for i in tqdm(range(0, len(images), batch_size), desc='Embedding images (CLIP local)'):
        batch = images[i:i+batch_size]
        embs = model.encode(batch, batch_size=batch_size, show_progress_bar=False,
                            normalize_embeddings=True, convert_to_numpy=True)
        all_embeddings.append(embs)
    return np.vstack(all_embeddings)


def embed_texts_clip(texts: list, model, batch_size: int = 64) -> np.ndarray:
    """
    Encode une liste de textes en embeddings via CLIP local.
    Retourne un tableau numpy (n_texts, EMBED_DIM), normalisé L2.
    """
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Embedding textes (CLIP local)'):
        batch = texts[i:i+batch_size]
        embs = model.encode(batch, batch_size=batch_size, show_progress_bar=False,
                            normalize_embeddings=True, convert_to_numpy=True)
        all_embeddings.append(embs)
    return np.vstack(all_embeddings)

print('Fonctions CLIP locales définies (image + texte).')


In [ ]:
# ── Paramètres ────────────────────────────────────────────────────────────────
CACHE_FILE_CORPUS = 'embeddings_vdr_corpus_clip.npy'
CACHE_FILE_IDS    = 'embeddings_vdr_corpus_ids.json'
BATCH_SIZE_IMG    = 16     # Batch réduit : ~200 Mo RAM par batch

# ── Sous-échantillonnage ───────────────────────────────────────────────────────
df_corpus_sample = df_corpus
print(f'Mode complet : {len(df_corpus)} pages')

corpus_ids_b    = df_corpus_sample['corpus_id'].tolist()
corpus_indices  = df_corpus_sample['corpus_idx'].tolist()


print('Encodage LAZY (batch par batch, images non accumulées en RAM)...')
all_embeddings = []

for i in tqdm(range(0, len(corpus_indices), BATCH_SIZE_IMG),
                desc='Encoding pages (lazy)'):
    batch_indices = corpus_indices[i:i+BATCH_SIZE_IMG]

    # Charger UNIQUEMENT ce batch depuis corpus_ds (lazy access)
    batch_pil = []
    for idx in batch_indices:
        img_field = corpus_ds[int(idx)]['image']
        batch_pil.append(to_pil(img_field))

    # Encoder le batch
    embs = clip_model.encode(
        batch_pil,
        batch_size=BATCH_SIZE_IMG,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    all_embeddings.append(embs)

    # Libérer la mémoire explicitement après chaque batch
    del batch_pil
    import gc; gc.collect()

corpus_embeddings_b = np.vstack(all_embeddings)

In [ ]:
cache = {
    cid: emb.tolist()
    for cid, emb in zip(corpus_ids_b, corpus_embeddings_b)
}
with open("embeddings_visual_corpus.json", "w") as f:
    json.dump(cache, f)

print(f"Sauvegardé : embeddings_visual_corpus.json ({len(cache)} entrées)")

In [ ]:
query_ids_b   = df_queries['query_id'].tolist()
query_texts_b = df_queries['query'].tolist()

print(f'Encodage de {len(query_texts_b)} requêtes textuelles via CLIP local...')
print('(troncature à 77 tokens — limite CLIP ViT-B/32)')

from transformers import CLIPTokenizerFast
tokenizer = CLIPTokenizerFast.from_pretrained('openai/clip-vit-base-patch32')

def truncate_for_clip(texts, max_tokens=77):
    """Tronque les textes à max_tokens via le tokenizer CLIP."""
    truncated = []
    for text in texts:
        tokens = tokenizer.encode(text)
        if len(tokens) > max_tokens:
            tokens = tokens[:max_tokens]
            text = tokenizer.decode(tokens, skip_special_tokens=True)
        truncated.append(text)
    return truncated

query_texts_truncated = truncate_for_clip(query_texts_b, max_tokens=77)

all_embeddings = []
BATCH_Q = 32  # batch réduit pour stabilité
for i in tqdm(range(0, len(query_texts_truncated), BATCH_Q),
                desc='Embedding requêtes (CLIP)'):
    batch = query_texts_truncated[i:i+BATCH_Q]
    embs = clip_model.encode(
        batch,
        batch_size=BATCH_Q,
        show_progress_bar=False,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    all_embeddings.append(embs)
    import gc; gc.collect()

query_embeddings_b = np.vstack(all_embeddings)

In [ ]:
cache = {
    cid: emb.tolist()
    for cid, emb in zip(query_ids_b, query_embeddings_b)
}
with open("embeddings_visual_queries.json", "w") as f:
    json.dump(cache, f)

print(f"Sauvegardé : embeddings_textual_queries.json ({len(cache)} entrées)")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_retrieval_results(query_embeddings, corpus_embeddings, query_ids, corpus_ids, k=10):
    results = {}
    batch_size = 256
    for i in tqdm(range(0, len(query_ids), batch_size), desc='Computing similarities'):
        batch_q_emb = query_embeddings[i:i+batch_size]
        batch_q_ids = query_ids[i:i+batch_size]
        sims = cosine_similarity(batch_q_emb, corpus_embeddings)  # (batch, corpus)
        top_k_indices = np.argsort(-sims, axis=1)[:, :k]
        for j, qid in enumerate(batch_q_ids):
            results[qid] = [
                (corpus_ids[idx], float(sims[j, idx]))
                for idx in top_k_indices[j]
            ]
    return results


def compute_ndcg_at_k(results, df_qrels, k=5):
    gt = defaultdict(dict)
    for _, row in df_qrels.iterrows():
        gt[row['query_id']][row['corpus_id']] = row['score']

    ndcg_scores = {}
    for qid, ranked_list in results.items():
        if qid not in gt:
            continue
        relevant = gt[qid]
        dcg = sum(
            (2**relevant.get(cid, 0) - 1) / np.log2(rank + 1)
            for rank, (cid, _) in enumerate(ranked_list[:k], start=1)
        )
        ideal_rels = sorted(relevant.values(), reverse=True)[:k]
        idcg = sum(
            (2**rel - 1) / np.log2(rank + 1)
            for rank, rel in enumerate(ideal_rels, start=1)
        )
        ndcg_scores[qid] = dcg / idcg if idcg > 0 else 0.0
    return ndcg_scores


print('Calcul des résultats de retrieval (Approche B — VDR Jina CLIP v2)...')
results_b = compute_retrieval_results(
    query_embeddings_b, corpus_embeddings_b, query_ids_b, corpus_ids_b, k=10
)
print(f'Résultats calculés pour {len(results_b):,} requêtes.')

# ── Filtrer les qrels sur les corpus_id disponibles (en cas de sous-échantillon) ──
corpus_ids_set = set(corpus_ids_b)
df_qrels_filtered = df_qrels[df_qrels['corpus_id'].isin(corpus_ids_set)]

ndcg_b = compute_ndcg_at_k(results_b, df_qrels_filtered, k=5)
mean_ndcg_b = np.mean(list(ndcg_b.values()))

print(f'\n{"="*55}')
print(f'  Approche B — VDR Jina CLIP v2 — nDCG@5 : {mean_ndcg_b:.4f}')
print(f'  Médiane                                  : {np.median(list(ndcg_b.values())):.4f}')
print(f'  % requêtes à 0.0                         : {sum(1 for v in ndcg_b.values() if v == 0) / len(ndcg_b) * 100:.1f}%')
print(f'  % requêtes à 1.0                         : {sum(1 for v in ndcg_b.values() if v == 1) / len(ndcg_b) * 100:.1f}%')
print(f'  N requêtes évaluées                      : {len(ndcg_b):,}')
print('='*55)

# Score de référence approche A
NDCG_A = 0.1029
delta = mean_ndcg_b - NDCG_A
print(f'\n  Delta vs Approche A (BioBERT) : {delta:+.4f} ({"gain" if delta > 0 else "perte"})')

In [ ]:
def extract_bboxes(raw) -> list:
    """
    Convertit le champ bounding_boxes en liste de dicts normalisés.
    ViDoRe V3 utilise le format : {'x1', 'y1', 'x2', 'y2'} en pixels absolus.
    """
    if raw is None:
        return []
    if isinstance(raw, np.ndarray):
        raw = raw.tolist()
    if not isinstance(raw, list):
        return []
    result = []
    for item in raw:
        if isinstance(item, np.ndarray):
            item = item.tolist()
        if isinstance(item, dict):
            # Format ViDoRe V3 : x1/y1/x2/y2 pixels absolus
            if 'x1' in item:
                result.append({
                    'x1': item['x1'], 'y1': item['y1'],
                    'x2': item['x2'], 'y2': item['y2']
                })
            # Format normalisé x/y/width/height (fallback)
            elif 'x' in item:
                result.append(item)
    return result


def draw_bounding_boxes(page_image: Image.Image, bboxes: list, title: str = '') -> None:
    """
    Affiche la page deux fois : brute puis avec bounding boxes.
    Gère le format x1/y1/x2/y2 en pixels absolus (ViDoRe V3).
    """
    w_img, h_img = page_image.size

    fig, axes = plt.subplots(1, 1, figsize=(16, 10))

    # ── Droite : page avec bboxes ─────────────────────────────────────────
    axes.imshow(page_image)
    colors = ['#FF4444', '#4488FF', '#44CC44']
    for i, bbox in enumerate(bboxes):
        color = colors[i % len(colors)]
        if 'x1' in bbox:
            # Pixels absolus
            x      = bbox['x1']
            y      = bbox['y1']
            width  = bbox['x2'] - bbox['x1']
            height = bbox['y2'] - bbox['y1']
        else:
            # Normalisé [0,1]
            x      = bbox.get('x', 0) * w_img
            y      = bbox.get('y', 0) * h_img
            width  = bbox.get('width', 0) * w_img
            height = bbox.get('height', 0) * h_img

        # Fallback simple pour la couleur de remplissage
        rect2 = patches.Rectangle(
            (x, y), width, height,
            linewidth=2.5, edgecolor=color, facecolor='none'
        )
        axes.add_patch(rect2)
        axes.text(x + 4, y + 18, f'Zone {i+1}',
                     color='white', fontsize=9, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

    axes.set_title('Zones annotées (qrels)', fontsize=11)
    axes.axis('off')

    fig.suptitle(title, fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()


# ── Index lazy : corpus_id → index dans corpus_ds ────────────────────────────
corpusid_to_idx = dict(zip(df_corpus['corpus_id'], df_corpus['corpus_idx']))

# ── Sélectionner les cas où VDR réussit ──────────────────────────────────────
good_cases = [(qid, score) for qid, score in ndcg_b.items() if score >= 0.8]
print(f'Requêtes avec nDCG@5 ≥ 0.8 : {len(good_cases)}')
if not good_cases:
    good_cases = sorted(ndcg_b.items(), key=lambda x: -x[1])[:3]
    print(f'Fallback — 3 meilleurs scores : {[f"{s:.3f}" for _, s in good_cases]}')

corpus_docid_dict = dict(zip(df_corpus['corpus_id'], df_corpus['doc_id']))
corpus_page_dict  = dict(zip(df_corpus['corpus_id'], df_corpus['page_number_in_doc']))
query_text_dict   = dict(zip(df_queries['query_id'], df_queries['query']))

# Construire le dict qrels avec bboxes
qrels_bbox_dict = defaultdict(list)
for _, row in df_qrels.iterrows():
    qrels_bbox_dict[(row['query_id'], row['corpus_id'])].append(
        extract_bboxes(row.get('bounding_boxes'))
    )

# ── Affichage des 3 meilleurs cas ─────────────────────────────────────────────
for qid, ndcg_score in sorted(good_cases, key=lambda x: -x[1])[:3]:
    query_text = query_text_dict.get(qid, 'N/A')
    top_cid, top_score = results_b[qid][0]

    all_bboxes = []
    for bbox_list in qrels_bbox_dict.get((qid, top_cid), []):
        all_bboxes.extend(bbox_list)

    doc_name = corpus_docid_dict.get(top_cid, '?')
    page_num = corpus_page_dict.get(top_cid, '?')

    print(f'\nnDCG@5={ndcg_score:.3f} | QID={qid}')
    print(f'  Requête  : {query_text}')
    print(f'  Page top1: {doc_name} p.{page_num} (cosinus={top_score:.3f})')
    print(f'  Bboxes   : {len(all_bboxes)} zone(s) → {all_bboxes}')

    idx = corpusid_to_idx.get(top_cid)
    if idx is not None:
        page_img = to_pil(corpus_ds[int(idx)]['image'])
        draw_bounding_boxes(
            page_img, all_bboxes,
            title=f'Q: "{query_text[:70]}"\n{doc_name} p.{page_num} | nDCG@5={ndcg_score:.3f}'
        )
        del page_img
        import gc; gc.collect()
    else:
        print('  Image non disponible dans le sous-échantillon.')


In [ ]:
worst_queries_b = sorted(ndcg_b.items(), key=lambda x: x[1])[:10]

gt_dict = defaultdict(dict)
for _, row in df_qrels.iterrows():
    gt_dict[row['query_id']][row['corpus_id']] = row['score']

corpus_meta_dict = dict(zip(
    df_corpus['corpus_id'],
    zip(df_corpus['doc_id'], df_corpus['page_number_in_doc'],
        df_corpus['markdown'].fillna('').str.len())
))

print('TOP 10 PIRES REQUÊTES — Approche B (VDR Jina CLIP v2)')
print('='*80)

worst_analysis_b = []
for qid, score in worst_queries_b:
    query_text = query_text_dict.get(qid, 'N/A')
    relevant_pages = gt_dict.get(qid, {})
    retrieved_top1 = results_b[qid][0][0] if results_b.get(qid) else None
    
    rel_info = []
    for cid, rel_score in relevant_pages.items():
        meta = corpus_meta_dict.get(cid, ('?', '?', 0))
        rel_info.append(f'{meta[0]} p.{meta[1]} (score={rel_score}, ocr_len={meta[2]})')
    
    print(f'\nnDCG@5={score:.3f} | QID: {qid}')
    print(f'  Requête   : {query_text}')
    print(f'  Attendu   : {" | ".join(rel_info[:2])}' + (' ...' if len(rel_info) > 2 else ''))
    if retrieved_top1:
        meta_r = corpus_meta_dict.get(retrieved_top1, ('?', '?', 0))
        print(f'  Récupéré  : {meta_r[0]} p.{meta_r[1]} (ocr_len={meta_r[2]})')
    
    worst_analysis_b.append({
        'query_id': qid, 'ndcg_b': score,
        'query': query_text,
        'expected_doc': rel_info[0] if rel_info else 'N/A'
    })

df_worst_b = pd.DataFrame(worst_analysis_b)
print('\n\nRésumé tableau des 10 pires requêtes B :')
display(df_worst_b[['query', 'ndcg_b', 'expected_doc']])